# 🧪 [Day 32 실전 핸즈온 워크북] 지식그래프(Knowledge Graph) 대용량 구축·적재 및 ETL 엔지니어링

> **학습 목표**:
> 1. **인코딩 정제**: CP949 공공데이터를 Python으로 감지하여 UTF-8로 표준화 변환
> 2. **제약조건 선행 생성**: `NODE KEY` 및 `UNIQUE` 제약조건으로 `MERGE` 연산 비용 $O(N^2) \rightarrow O(\log N)$ 최적화
> 3. **LOAD CSV 정형 적재**: `toInteger`, `toFloat`, `coalesce`, `trim`을 통한 결측치 방어 및 멱등성 보장
> 4. **apoc.load.json 계층 분해**: 중첩 JSON 배열을 `UNWIND`하여 노드와 관계를 원자적으로 매핑
> 5. **배치 트랜잭션 최적화**: `CALL { ... } IN TRANSACTIONS OF 1000 ROWS`로 대용량 데이터 적재 시 OOM 방지
> 6. **메타 통계 검증**: `apoc.meta.stats()`를 활용한 엔터프라이즈 그래프 정합성 점검

## 0. 환경 준비 및 드라이버 연결

In [ ]:
import os
import json
import shutil
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 환경변수 로드
load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 1. CP949 공공데이터 UTF-8 변환 및 import 폴더 배치

In [ ]:
src_dir = Path('data') if Path('data').exists() else Path('../data')

# 1) CP949 -> UTF-8 변환
cp949_files = [
    ('seoul_metro_transfer_cp949.csv', 'seoul_metro_transfer_utf8.csv'),
    ('seoul_metro_stations_cp949.csv', 'seoul_metro_stations_utf8.csv')
]

for src, tgt in cp949_files:
    p_src = src_dir / src
    p_tgt = src_dir / tgt
    if p_src.exists():
        text = p_src.read_text(encoding='cp949')
        p_tgt.write_text(text, encoding='utf-8')
        print(f'✅ UTF-8 변환 완료: {tgt}')

# 2) import 폴더 복사
if NEO4J_IMPORT_DIR and Path(NEO4J_IMPORT_DIR).exists():
    imp_path = Path(NEO4J_IMPORT_DIR)
    for f in src_dir.glob('*.*'):
        shutil.copy2(f, imp_path / f.name)
    print('✅ import 폴더 복사 완료:', imp_path)

## 2. 스키마 제약조건 선행 배포 (NODE KEY & UNIQUE)

In [ ]:
# 제약조건 생성
constraints = [
    "CREATE CONSTRAINT metro_station_key IF NOT EXISTS FOR (s:Station) REQUIRE (s.name, s.line) IS NODE KEY",
    "CREATE CONSTRAINT metro_line_id IF NOT EXISTS FOR (l:Line) REQUIRE l.line_id IS UNIQUE",
    "CREATE CONSTRAINT metro_day_name IF NOT EXISTS FOR (d:Day) REQUIRE d.name IS UNIQUE"
]

for c_q in constraints:
    try:
        run_cypher(c_q)
        print('✅ 제약조건 생성 성공:', c_q.split()[2])
    except Exception as e:
        print('ℹ️ 제약조건 생성 주의:', e)

print("\n[현재 활성 제약조건 목록]")
for row in run_cypher("SHOW CONSTRAINTS YIELD name, type, entityType RETURN name, type, entityType"):
    print("  •", row)

## 3. 요일(:Day) 노드 생성 및 LOAD CSV 환승 데이터 적재

In [ ]:
# 1) 요일 노드 생성
day_q = """
UNWIND [
    {name: '월요일', num: 1, weekend: false},
    {name: '화요일', num: 2, weekend: false},
    {name: '수요일', num: 3, weekend: false},
    {name: '목요일', num: 4, weekend: false},
    {name: '금요일', num: 5, weekend: false},
    {name: '토요일', num: 6, weekend: true},
    {name: '일요일', num: 7, weekend: true}
] AS row
MERGE (d:Day {name: row.name})
SET d.day_num = row.num, d.is_weekend = row.weekend
RETURN count(d) AS day_count
"""
res_d = run_cypher(day_q)
print('✅ 요일 노드 생성:', res_d[0]['day_count'], '개')

# 2) 환승 데이터 LOAD CSV 적재
load_csv_q = """
LOAD CSV WITH HEADERS FROM 'file:///seoul_metro_transfer_utf8.csv' AS row
WITH row
WHERE row.역명 IS NOT NULL AND row.호선 IS NOT NULL
MERGE (s:Station {name: trim(row.역명), line: trim(row.호선)})
SET s.is_transfer = true
WITH s, row
MATCH (d:Day {name: trim(row.요일)})
MERGE (s)-[r:HAS_TRANSFER_STAT]->(d)
SET r.passengers = toInteger(coalesce(row.환승인원, '0'))
RETURN count(r) AS rel_count
"""
try:
    res_csv = run_cypher(load_csv_q)
    print('✅ 환승 통계 관계 적재:', res_csv[0]['rel_count'], '건')
except Exception as e:
    print('⚠️ LOAD CSV 실행 중 알림:', e)

## 4. apoc.load.json 기반 노선망(:Line) 및 소속(:BELONGS_TO) 적재

In [ ]:
load_json_q = """
CALL apoc.load.json('file:///seoul_metro_lines.json') YIELD value
UNWIND value.lines AS line_data
MERGE (l:Line {line_id: line_data.line_id})
SET l.name = line_data.line_name,
    l.color = line_data.color
WITH line_data, l
UNWIND line_data.stations AS st_name
MERGE (s:Station {name: st_name, line: line_data.line_name})
MERGE (s)-[:BELONGS_TO]->(l)
RETURN count(l) AS line_count
"""
try:
    res_json = run_cypher(load_json_q)
    print('✅ JSON 노선 및 소속 관계 적재 완료!')
except Exception as e:
    print('⚠️ apoc.load.json 실행 중 알림:', e)

## 5. 지식그래프 검증 및 통계 조회

In [ ]:
# 1. 노드 레이블별 카운트
nodes = run_cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC")
print("📊 [노드 레이블 통계]")
for n in nodes:
    print(f"  • :{n['label']}: {n['cnt']:,}개")

# 2. 관계 타입별 카운트
rels = run_cypher("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS cnt ORDER BY cnt DESC")
print("\n📊 [관계 타입 통계]")
for r in rels:
    print(f"  • [:{r['type']}]: {r['cnt']:,}건")

# 3. 환승인원 Top 5 역 조회 (파라미터 바인딩)
top_transfer = run_cypher("""
MATCH (s:Station)-[r:HAS_TRANSFER_STAT]->(d:Day {name: $day_name})
RETURN s.name AS 역명, s.line AS 호선, r.passengers AS 환승인원
ORDER BY 환승인원 DESC LIMIT 5
""", day_name='월요일')
print("\n🏆 [월요일 환승인원 Top 5 역]")
for idx, st in enumerate(top_transfer, 1):
    print(f"  {idx}. {st['역명']} ({st['호선']}): {st['환승인원']:,}명")